# Notebook 4: Model Monitoring & Drift Detection

Set up ML Observability in PROD, simulate feature drift (600k inference records), query drift and performance metrics, and configure automated retraining triggers.

**Pipeline mapping**: Replaces SageMaker Model Monitor + CloudWatch + EventBridge (5 services -> 2 Snowflake objects)

**Cost advantage**: No CloudWatch metric storage costs, no EventBridge rule costs, no separate SageMaker Model Monitor scheduling job. Model Monitor uses existing warehouse (auto-suspend between refreshes). Alert checks run on-demand, not always-on infrastructure.

In [ ]:
# Set up session pointing to PD_DEMO_PROD.MONITORING schema
# This is where inference logs, baseline data, and monitor objects live
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

session = get_active_session()
session.use_database('PD_DEMO_PROD')
session.use_schema('MONITORING')

## 1. Create Inference Log Table

In production, this is populated by the serving endpoint (auto-capture) or by your application logging pipeline. We simulate data with two periods: normal and drifted.

In [ ]:
-- =============================================================================
-- CREATE INFERENCE LOG TABLE: schema for storing all production predictions
-- Columns: prediction_id, timestamp, all 20 model features, origination_channel,
--   predicted_pd (model output), actual_default_90dpm (ground truth), model_version
-- In production, SPCS auto-capture can populate this automatically
-- No separate S3 capture bucket or CloudWatch data quality constraints needed
-- =============================================================================
CREATE OR REPLACE TABLE PD_DEMO_PROD.MONITORING.INFERENCE_LOG (
    PREDICTION_ID STRING,
    PREDICTION_TIMESTAMP TIMESTAMP_NTZ,
    NUM_CREDIT_PRODUCTS NUMBER,
    NUM_INACTIVE_ACCOUNTS NUMBER,
    NUM_CREDIT_SEARCHES_L6M NUMBER,
    TOTAL_OUTSTANDING_BALANCE FLOAT,
    MAX_DELINQUENCY_DAYS NUMBER,
    CREDIT_UTILISATION_RATIO FLOAT,
    MONTHS_SINCE_OLDEST_ACCOUNT NUMBER,
    NUM_DEFAULTS_L3Y NUMBER,
    NUM_CCJS NUMBER,
    CREDIT_SCORE NUMBER,
    NUM_OPEN_ACCOUNTS NUMBER,
    TOTAL_CREDIT_LIMIT FLOAT,
    NUM_MISSED_PAYMENTS_L12M NUMBER,
    DEBT_TO_INCOME_RATIO FLOAT,
    MONTHS_SINCE_LAST_DEFAULT NUMBER,
    NUM_HARD_SEARCHES_L3M NUMBER,
    APPLICANT_AGE_YEARS NUMBER,
    CHANNEL_DIRECT NUMBER,
    CHANNEL_GOOGLE NUMBER,
    CHANNEL_META NUMBER,
    ORIGINATION_CHANNEL STRING,
    PREDICTED_PD FLOAT,
    ACTUAL_DEFAULT_90DPM NUMBER,
    MODEL_VERSION STRING
);

In [ ]:
-- =============================================================================
-- INSERT 300k NORMAL-PERIOD PREDICTIONS (30-60 days ago, ~10k/day)
-- Feature distributions match training data (same NORMAL() parameters as nb01)
-- credit_score ~ N(650, 80), utilisation ~ N(0.35, 0.2), ~6% default rate
-- Channel mix: Direct 40% / Google 35% / Meta 25% (matches training)
-- predicted_pd: uniform 0.02-0.15 (stable, well-calibrated predictions)
-- This period represents steady-state production performance (no drift)
-- =============================================================================
INSERT INTO PD_DEMO_PROD.MONITORING.INFERENCE_LOG
WITH normal_data AS (
    SELECT
        'PRED-N-' || LPAD(SEQ4()::STRING, 8, '0') AS prediction_id,
        DATEADD('minute', -UNIFORM(43200, 86400, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ AS prediction_timestamp,
        GREATEST(0, ROUND(NORMAL(4, 2.5, RANDOM()))) AS num_credit_products,
        GREATEST(0, ROUND(NORMAL(1.5, 1.2, RANDOM()))) AS num_inactive_accounts,
        GREATEST(0, ROUND(NORMAL(3, 2, RANDOM()))) AS num_credit_searches_l6m,
        GREATEST(0, ROUND(NORMAL(8500, 6000, RANDOM()), 2)) AS total_outstanding_balance,
        GREATEST(0, ROUND(NORMAL(5, 15, RANDOM()))) AS max_delinquency_days,
        LEAST(1.0, GREATEST(0, ROUND(NORMAL(0.35, 0.2, RANDOM()), 3))) AS credit_utilisation_ratio,
        GREATEST(6, ROUND(NORMAL(84, 48, RANDOM()))) AS months_since_oldest_account,
        GREATEST(0, ROUND(NORMAL(0.3, 0.7, RANDOM()))) AS num_defaults_l3y,
        GREATEST(0, ROUND(NORMAL(0.1, 0.4, RANDOM()))) AS num_ccjs,
        GREATEST(300, LEAST(850, ROUND(NORMAL(650, 80, RANDOM())))) AS credit_score,
        GREATEST(0, ROUND(NORMAL(3, 2, RANDOM()))) AS num_open_accounts,
        GREATEST(0, ROUND(NORMAL(15000, 10000, RANDOM()), 2)) AS total_credit_limit,
        GREATEST(0, ROUND(NORMAL(0.5, 1.2, RANDOM()))) AS num_missed_payments_l12m,
        LEAST(2.0, GREATEST(0, ROUND(NORMAL(0.35, 0.2, RANDOM()), 3))) AS debt_to_income_ratio,
        GREATEST(0, ROUND(NORMAL(36, 30, RANDOM()))) AS months_since_last_default,
        GREATEST(0, ROUND(NORMAL(1.5, 1.5, RANDOM()))) AS num_hard_searches_l3m,
        GREATEST(18, LEAST(75, ROUND(NORMAL(32, 8, RANDOM())))) AS applicant_age_years,
        CASE WHEN UNIFORM(1,100,RANDOM()) <= 40 THEN 'Direct'
             WHEN UNIFORM(1,100,RANDOM()) <= 75 THEN 'Google'
             ELSE 'Meta' END AS origination_channel,
        UNIFORM(0.02, 0.15, RANDOM())::FLOAT AS predicted_pd,
        CASE WHEN UNIFORM(0,100,RANDOM()) < 6 THEN 1 ELSE 0 END AS actual_default_90dpm,
        'V1' AS model_version
    FROM TABLE(GENERATOR(ROWCOUNT => 300000))
)
SELECT
    prediction_id, prediction_timestamp,
    num_credit_products, num_inactive_accounts, num_credit_searches_l6m,
    total_outstanding_balance, max_delinquency_days, credit_utilisation_ratio,
    months_since_oldest_account, num_defaults_l3y, num_ccjs, credit_score,
    num_open_accounts, total_credit_limit, num_missed_payments_l12m,
    debt_to_income_ratio, months_since_last_default, num_hard_searches_l3m,
    applicant_age_years,
    CASE WHEN origination_channel='Direct' THEN 1 ELSE 0 END,
    CASE WHEN origination_channel='Google' THEN 1 ELSE 0 END,
    CASE WHEN origination_channel='Meta' THEN 1 ELSE 0 END,
    origination_channel, predicted_pd, actual_default_90dpm, model_version
FROM normal_data;

In [ ]:
-- =============================================================================
-- INSERT 300k DRIFTED-PERIOD PREDICTIONS (last 30 days, ~10k/day)
-- Simulates a macro-economic downturn affecting the applicant population:
--   credit_score: N(580, 90) vs training N(650, 80)  -- 70pt drop
--   utilisation:  N(0.50, 0.25) vs training N(0.35, 0.2) -- 15pp increase
--   outstanding_balance: N(12k, 7k) vs training N(8.5k, 6k)
--   missed_payments: N(1.5, 2.0) vs training N(0.5, 1.2)
--   debt_to_income: N(0.55, 0.25) vs training N(0.35, 0.2)
--   applicant_age: N(27, 6) vs training N(32, 8) -- younger, riskier cohort
-- Channel mix shifts: Direct 20% / Google 30% / Meta 50% (marketing reallocation)
-- Default rate rises to ~12% (double the training rate of ~6%)
-- predicted_pd: 0.05-0.35 (higher, reflecting model uncertainty)
-- This is the data that should trigger PSI alerts in the Model Monitor
-- =============================================================================
INSERT INTO PD_DEMO_PROD.MONITORING.INFERENCE_LOG
WITH drifted_data AS (
    SELECT
        'PRED-D-' || LPAD(SEQ4()::STRING, 8, '0') AS prediction_id,
        DATEADD('minute', -UNIFORM(0, 43200, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ AS prediction_timestamp,
        GREATEST(0, ROUND(NORMAL(5, 3, RANDOM()))) AS num_credit_products,
        GREATEST(0, ROUND(NORMAL(2, 1.5, RANDOM()))) AS num_inactive_accounts,
        GREATEST(0, ROUND(NORMAL(5, 3, RANDOM()))) AS num_credit_searches_l6m,
        GREATEST(0, ROUND(NORMAL(12000, 7000, RANDOM()), 2)) AS total_outstanding_balance,
        GREATEST(0, ROUND(NORMAL(10, 20, RANDOM()))) AS max_delinquency_days,
        LEAST(1.0, GREATEST(0, ROUND(NORMAL(0.50, 0.25, RANDOM()), 3))) AS credit_utilisation_ratio,
        GREATEST(6, ROUND(NORMAL(60, 36, RANDOM()))) AS months_since_oldest_account,
        GREATEST(0, ROUND(NORMAL(0.8, 1.0, RANDOM()))) AS num_defaults_l3y,
        GREATEST(0, ROUND(NORMAL(0.3, 0.6, RANDOM()))) AS num_ccjs,
        GREATEST(300, LEAST(850, ROUND(NORMAL(580, 90, RANDOM())))) AS credit_score,
        -- Drifted new features: more open accounts, higher missed payments, younger applicants
        GREATEST(0, ROUND(NORMAL(5, 3, RANDOM()))) AS num_open_accounts,
        GREATEST(0, ROUND(NORMAL(18000, 12000, RANDOM()), 2)) AS total_credit_limit,
        GREATEST(0, ROUND(NORMAL(1.5, 2.0, RANDOM()))) AS num_missed_payments_l12m,
        LEAST(2.0, GREATEST(0, ROUND(NORMAL(0.55, 0.25, RANDOM()), 3))) AS debt_to_income_ratio,
        GREATEST(0, ROUND(NORMAL(18, 20, RANDOM()))) AS months_since_last_default,
        GREATEST(0, ROUND(NORMAL(3, 2, RANDOM()))) AS num_hard_searches_l3m,
        GREATEST(18, LEAST(75, ROUND(NORMAL(27, 6, RANDOM())))) AS applicant_age_years,
        -- Channel mix shift: more Meta, less Direct (marketing budget reallocation)
        CASE WHEN UNIFORM(1,100,RANDOM()) <= 20 THEN 'Direct'
             WHEN UNIFORM(1,100,RANDOM()) <= 50 THEN 'Google'
             ELSE 'Meta' END AS origination_channel,
        UNIFORM(0.05, 0.35, RANDOM())::FLOAT AS predicted_pd,
        CASE WHEN UNIFORM(0,100,RANDOM()) < 12 THEN 1 ELSE 0 END AS actual_default_90dpm,
        'V1' AS model_version
    FROM TABLE(GENERATOR(ROWCOUNT => 300000))
)
SELECT
    prediction_id, prediction_timestamp,
    num_credit_products, num_inactive_accounts, num_credit_searches_l6m,
    total_outstanding_balance, max_delinquency_days, credit_utilisation_ratio,
    months_since_oldest_account, num_defaults_l3y, num_ccjs, credit_score,
    num_open_accounts, total_credit_limit, num_missed_payments_l12m,
    debt_to_income_ratio, months_since_last_default, num_hard_searches_l3m,
    applicant_age_years,
    CASE WHEN origination_channel='Direct' THEN 1 ELSE 0 END,
    CASE WHEN origination_channel='Google' THEN 1 ELSE 0 END,
    CASE WHEN origination_channel='Meta' THEN 1 ELSE 0 END,
    origination_channel, predicted_pd, actual_default_90dpm, model_version
FROM drifted_data;

In [ ]:
-- Verify: total prediction count (expect 600k), date range, avg PD score, actual default rate
-- The avg_pd and default_rate should be higher than training due to the drifted period
SELECT
    COUNT(*) AS total_predictions,
    MIN(prediction_timestamp) AS earliest,
    MAX(prediction_timestamp) AS latest,
    ROUND(AVG(predicted_pd), 4) AS avg_pd_score,
    ROUND(AVG(actual_default_90dpm) * 100, 2) AS actual_default_rate_pct
FROM PD_DEMO_PROD.MONITORING.INFERENCE_LOG;

## 2. Create Baseline Table

The baseline represents the expected feature distributions from training time. Used by the Model Monitor for drift calculation (PSI).

In [ ]:
-- Create baseline table from the NORMAL period only (>30 days ago)
-- This is the reference distribution the Model Monitor compares against
-- Contains: prediction timestamp, all features, channel, predicted_pd, actual label
-- In SageMaker, baseline data lives in S3 as JSON constraint files
CREATE OR REPLACE TABLE PD_DEMO_PROD.MONITORING.BASELINE_DATA AS
SELECT
    PREDICTION_TIMESTAMP, NUM_CREDIT_PRODUCTS, NUM_INACTIVE_ACCOUNTS,
    NUM_CREDIT_SEARCHES_L6M, TOTAL_OUTSTANDING_BALANCE, MAX_DELINQUENCY_DAYS,
    CREDIT_UTILISATION_RATIO, MONTHS_SINCE_OLDEST_ACCOUNT, NUM_DEFAULTS_L3Y,
    NUM_CCJS, CREDIT_SCORE, CHANNEL_DIRECT, CHANNEL_GOOGLE, CHANNEL_META,
    ORIGINATION_CHANNEL, PREDICTED_PD, ACTUAL_DEFAULT_90DPM
FROM PD_DEMO_PROD.MONITORING.INFERENCE_LOG
WHERE PREDICTION_TIMESTAMP < DATEADD('day', -30, CURRENT_TIMESTAMP());

## 3. Create Model Monitor

Replaces SageMaker Model Monitor + CloudWatch metrics configuration. One SQL statement sets up drift detection, performance tracking, and segmentation.

In [ ]:
-- =============================================================================
-- CREATE MODEL MONITOR on the production model (PD_XGBOOST V1)
-- Configuration:
--   SOURCE:             INFERENCE_LOG (600k predictions)
--   BASELINE:           BASELINE_DATA (300k normal-period predictions)
--   REFRESH_INTERVAL:   1 day (can be adjusted to hourly)
--   AGGREGATION_WINDOW: 1 day (calculates metrics per day)
--   PREDICTION_SCORE:   PREDICTED_PD column
--   ACTUAL_SCORE:       ACTUAL_DEFAULT_90DPM column (for performance metrics)
--   SEGMENT_COLUMNS:    ORIGINATION_CHANNEL (drift tracked per channel)
-- This single SQL statement replaces:
--   1. SageMaker Model Monitor configuration
--   2. CloudWatch metric namespace setup
--   3. S3 baseline constraint JSON files
-- =============================================================================
USE SCHEMA PD_DEMO_PROD.REGISTRY;

CREATE MODEL MONITOR IF NOT EXISTS PD_DEMO_PROD.REGISTRY.PD_DRIFT_MONITOR WITH
    MODEL = PD_XGBOOST
    VERSION = 'V1'
    FUNCTION = 'predict_proba'
    SOURCE = PD_DEMO_PROD.MONITORING.INFERENCE_LOG
    BASELINE = PD_DEMO_PROD.MONITORING.BASELINE_DATA
    WAREHOUSE = PD_DEMO_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_PD')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL_DEFAULT_90DPM')
    SEGMENT_COLUMNS = ('ORIGINATION_CHANNEL');

In [ ]:
-- Verify monitor configuration: shows all settings (source, baseline, segments, refresh interval)
DESC MODEL MONITOR PD_DEMO_PROD.REGISTRY.PD_DRIFT_MONITOR;

## 4. Query Drift Metrics

PSI (Population Stability Index) quantifies how much a feature's distribution has shifted from the baseline. PSI > 0.25 typically indicates significant drift requiring investigation.

In [ ]:
# Query PSI (Population Stability Index) for 8 key features over the last 60 days
# PSI thresholds: <0.10 = stable, 0.10-0.25 = moderate drift, >0.25 = significant drift
# Features monitored: credit_score, total_outstanding_balance, credit_utilisation_ratio,
#   num_credit_searches_l6m, max_delinquency_days, debt_to_income_ratio,
#   num_missed_payments_l12m, applicant_age_years
# Uses MODEL_MONITOR_DRIFT_METRIC() function -- no CloudWatch dashboard needed
drift_features = ['CREDIT_SCORE', 'TOTAL_OUTSTANDING_BALANCE', 'CREDIT_UTILISATION_RATIO',
                   'NUM_CREDIT_SEARCHES_L6M', 'MAX_DELINQUENCY_DAYS',
                   'DEBT_TO_INCOME_RATIO', 'NUM_MISSED_PAYMENTS_L12M', 'APPLICANT_AGE_YEARS']

drift_results = []
for feat in drift_features:
    try:
        result = session.sql(f"""
            SELECT * FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
                'PD_DEMO_PROD.REGISTRY.PD_DRIFT_MONITOR',
                'PSI',
                '{feat}',
                'DAY',
                DATEADD('day', -60, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
                CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
            ))
        """).to_pandas()
        result['FEATURE'] = feat
        drift_results.append(result)
    except Exception as e:
        print(f"Note: {feat} - {e}")

if drift_results:
    drift_df = pd.concat(drift_results, ignore_index=True)
    print(f"Drift metrics retrieved for {len(drift_results)} features")
    drift_df.head(20)
else:
    print("Monitor is still initialising. Drift metrics will be available after the first refresh.")

In [ ]:
# Plot PSI drift results (if available after monitor refresh)
# Left panel:  PSI over time per feature (line chart) -- shows when drift started
# Right panel: latest PSI per feature (horizontal bar) -- colour-coded by severity
#   Green = stable (<0.10), Orange = warning (0.10-0.25), Red = alert (>0.25)
# Expected: credit_score and utilisation should show significant drift (>0.25)
if drift_results and len(drift_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    pivot = drift_df.pivot_table(index='TIMESTAMP', columns='FEATURE', values='VALUE', aggfunc='mean')
    pivot.plot(ax=axes[0], marker='o', linewidth=2)
    axes[0].axhline(y=0.25, color='red', linestyle='--', label='Alert threshold (PSI=0.25)')
    axes[0].axhline(y=0.10, color='orange', linestyle='--', label='Warning threshold (PSI=0.10)')
    axes[0].set_title('PSI Over Time by Feature')
    axes[0].set_ylabel('PSI Value')
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis='x', rotation=45)

    latest = drift_df.groupby('FEATURE')['VALUE'].last().sort_values(ascending=True)
    colors = ['red' if v > 0.25 else 'orange' if v > 0.10 else 'green' for v in latest.values]
    latest.plot(kind='barh', ax=axes[1], color=colors)
    axes[1].axvline(x=0.25, color='red', linestyle='--', label='Alert')
    axes[1].axvline(x=0.10, color='orange', linestyle='--', label='Warning')
    axes[1].set_title('Latest PSI by Feature')
    axes[1].set_xlabel('PSI')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Visualisation will be available after monitor refreshes.")

## 5. Performance Metrics by Segment

Track AUC over time, segmented by origination channel. This catches degradation in specific cohorts that overall metrics might miss.

In [ ]:
# Query AUC performance metric segmented by origination channel (Direct/Google/Meta)
# Uses MODEL_MONITOR_PERFORMANCE_METRIC() with SEGMENTS filter
# Shows whether model performance degrades differently across channels
# Key insight: Meta channel likely degrades more (channel mix shifted in drifted period)
# In SageMaker, this segmented analysis requires custom code per segment
perf_results = []
for channel in ['Direct', 'Google', 'Meta']:
    try:
        result = session.sql(f"""
            SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
                'PD_DEMO_PROD.REGISTRY.PD_DRIFT_MONITOR',
                'AUC',
                'DAY',
                DATEADD('day', -60, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
                CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
                '{{"SEGMENTS": [{{"column": "ORIGINATION_CHANNEL", "value": "{channel}"}}]}}'
            ))
        """).to_pandas()
        result['CHANNEL'] = channel
        perf_results.append(result)
    except Exception as e:
        print(f"Note: {channel} - {e}")

if perf_results:
    perf_df = pd.concat(perf_results, ignore_index=True)
    print("Performance metrics by channel:")
    perf_df.head(20)
else:
    print("Performance metrics will be available once ground truth labels are populated and monitor refreshes.")

## 6. Manual Drift Analysis (Feature Distribution Comparison)

Even before the monitor's first refresh, we can directly compare distributions between periods to visualise the drift we injected.

In [ ]:
# Visual drift analysis: overlay histograms comparing normal vs drifted periods
# Plots 8 features: credit_score, outstanding_balance, utilisation, searches,
#   debt_to_income, missed_payments, applicant_age, predicted_pd
# Also prints summary statistics (mean, std) by period to quantify the shift
# Expected: clear distribution shift visible in credit_score (~650 -> ~580),
#   utilisation (~0.35 -> ~0.50), and predicted_pd (upward shift)
log_df = session.table('PD_DEMO_PROD.MONITORING.INFERENCE_LOG').to_pandas()
log_df['PERIOD'] = np.where(
    log_df['PREDICTION_TIMESTAMP'] < (pd.Timestamp.now() - pd.Timedelta(days=30)),
    'Normal (30-60d ago)', 'Drifted (last 30d)'
)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
drift_cols = ['CREDIT_SCORE', 'TOTAL_OUTSTANDING_BALANCE', 'CREDIT_UTILISATION_RATIO',
              'NUM_CREDIT_SEARCHES_L6M', 'DEBT_TO_INCOME_RATIO', 'NUM_MISSED_PAYMENTS_L12M',
              'APPLICANT_AGE_YEARS', 'PREDICTED_PD']

for idx, c in enumerate(drift_cols):
    ax = axes[idx // 4, idx % 4]
    for period, group in log_df.groupby('PERIOD'):
        ax.hist(group[c], bins=40, alpha=0.5, label=period, density=True)
    ax.set_title(c.replace('_', ' ').title(), fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle('Feature Distribution: Normal vs Drifted Period (600k records)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\nSummary statistics by period:")
summary = log_df.groupby('PERIOD')[drift_cols].agg(['mean', 'std']).round(3)
print(summary.to_string())

## 7. Automated Retraining: Drift-Triggered + Scheduled

Replaces CloudWatch Alarms + EventBridge Rules + SageMaker Pipeline triggers. Two retraining modes:
- **7a. Drift-triggered**: Snowflake Alert monitors PSI and fires a Task when threshold exceeded (replaces CloudWatch Alarm → EventBridge → SM Pipeline)
- **7b. Scheduled**: A separate Task runs on a fixed cadence (replaces EventBridge scheduled rule → SM Pipeline)

In [ ]:
-- =============================================================================
-- AUTOMATED RETRAINING: Alert + Task replaces CloudWatch + EventBridge + SM Pipelines
-- Cost: Alert runs on existing warehouse (auto-suspend between checks)
-- No always-on EventBridge rules or CloudWatch alarm evaluation costs
-- Performance: Task triggers immediately on drift detection, not on a polling cycle
-- =============================================================================

-- Task: Automated retraining pipeline (stub -- in production this calls a stored procedure)
CREATE OR REPLACE TASK PD_DEMO_PROD.MONITORING.PD_RETRAIN_TASK
    WAREHOUSE = PD_DEMO_WH
    AS
    BEGIN
        -- In production, this stored procedure would:
        -- 1. Pull latest training data from Feature Store in DEV
        -- 2. Retrain XGBoost with the same HPO config
        -- 3. Evaluate on held-out set
        -- 4. If metrics pass threshold, promote through DEV -> STAGING -> PROD
        -- 5. Redeploy the SPCS service with the new model version
        CALL SYSTEM$LOG_TRACE('PD model retraining triggered by drift alert');
    END;

-- Alert: fires when PSI exceeds threshold on any monitored feature
-- Runs every 6 hours using existing warehouse (auto-suspends between runs)
CREATE OR REPLACE ALERT PD_DEMO_PROD.MONITORING.PD_DRIFT_ALERT
    WAREHOUSE = PD_DEMO_WH
    SCHEDULE = 'USING CRON 0 */6 * * * UTC'
    IF (EXISTS (
        SELECT 1 FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'PD_DEMO_PROD.REGISTRY.PD_DRIFT_MONITOR',
            'PSI',
            'CREDIT_SCORE',
            'DAY',
            DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
        WHERE VALUE > 0.25
    ))
    THEN EXECUTE TASK PD_DEMO_PROD.MONITORING.PD_RETRAIN_TASK;

In [ ]:
print("""
=== Automated Retraining Architecture ===

Current (SageMaker):
  SM Model Monitor --> CloudWatch Metrics --> CloudWatch Alarm --> EventBridge Rule --> SM Pipeline
  (5 services to configure and wire together)

Proposed (Snowflake):
  Model Monitor --> Snowflake Alert --> Snowflake Task
  (all native, single platform, single set of permissions)

Key advantages:
  1. No cross-service IAM wiring
  2. Alert condition is a SQL query (PSI > threshold)
  3. Task can call any stored procedure or notebook
  4. Full audit trail in Snowflake query history
  5. RBAC governs who can create/modify alerts and tasks
""")

### 7b. Scheduled Retraining (independent of drift)

In addition to drift-triggered retraining, production ML pipelines often retrain on a fixed cadence (e.g. weekly or monthly) to incorporate the latest ground truth labels. In SageMaker this requires a separate EventBridge scheduled rule wired to the pipeline. In Snowflake, it is a single Task with a CRON schedule.

In [ ]:
-- =============================================================================
-- SCHEDULED RETRAINING: Fixed-cadence retrain replaces EventBridge scheduled rule
-- SageMaker requires: EventBridge Rule (cron) -> Step Functions -> SM Pipeline
-- Snowflake: one Task object with a CRON schedule
-- Cost: runs on existing warehouse with auto-suspend -- no always-on scheduler
-- =============================================================================

-- Weekly scheduled retrain (every Sunday at 02:00 UTC)
-- This runs independently of drift alerts -- ensures the model stays fresh
-- even if PSI stays below threshold (e.g. gradual concept drift)
CREATE OR REPLACE TASK PD_DEMO_PROD.MONITORING.PD_SCHEDULED_RETRAIN
    WAREHOUSE = PD_DEMO_WH
    SCHEDULE = 'USING CRON 0 2 * * 0 UTC'
    AS
    BEGIN
        -- In production, this calls the same retraining stored procedure:
        -- 1. Pull latest 6 months of labelled data from Feature Store
        -- 2. Retrain with same HPO config, evaluate on held-out set
        -- 3. Compare metrics against current production model
        -- 4. If improved: promote through DEV -> STAGING -> PROD
        -- 5. If not improved: log result and skip promotion
        CALL SYSTEM$LOG_TRACE('PD model scheduled weekly retraining started');
    END;

-- Both retraining mechanisms can coexist:
--   PD_DRIFT_ALERT   -> PD_RETRAIN_TASK       (reactive: respond to drift)
--   PD_SCHEDULED_RETRAIN                       (proactive: weekly refresh)
-- SageMaker requires separate EventBridge rules for each trigger type
-- Snowflake: just create another Task -- same warehouse, same RBAC, same audit trail

## 8. SageMaker Monitor vs Snowflake ML Observability

In [ ]:
comparison = pd.DataFrame({
    'Capability': [
        'Drift detection',
        'Performance tracking',
        'Segmentation',
        'Alerting',
        'Retraining trigger',
        'Dashboard',
        'Data location',
        'Services to manage',
        'Configuration'
    ],
    'SageMaker Model Monitor': [
        'KL divergence, KS test, Chi-squared',
        'Custom CloudWatch metrics',
        'Custom analysis code needed',
        'CloudWatch Alarms (separate config)',
        'EventBridge -> SM Pipeline (separate config)',
        'CloudWatch Dashboard (separate config)',
        'S3 (baseline + captured data)',
        'SM Monitor + CloudWatch + EventBridge + S3',
        'JSON baseline constraints + multiple AWS consoles'
    ],
    'Snowflake ML Observability': [
        'PSI, KL divergence',
        'AUC, F1, precision, recall, MAE, RMSE',
        'Built-in (up to 5 segment columns)',
        'Snowflake Alerts (same platform)',
        'Alert -> Task (same platform)',
        'Snowsight Model Monitor UI',
        'Snowflake tables (no data movement)',
        'Model Monitor only (single object)',
        'One SQL statement (CREATE MODEL MONITOR)'
    ]
})

print(comparison.to_string(index=False))

---
## Summary

| What we did | SageMaker equivalent | Snowflake approach |
|---|---|---|
| Create inference log | S3 capture + data quality constraints | Snowflake table (auto-capture available) |
| Set up drift monitoring | SM Model Monitor + S3 baseline | `CREATE MODEL MONITOR` (1 SQL statement) |
| Query PSI per feature | Custom CloudWatch metrics | `MODEL_MONITOR_DRIFT_METRIC()` function |
| Segment by channel | Custom analysis code | Built-in `SEGMENT_COLUMNS` parameter |
| Drift-triggered retraining | CloudWatch Alarm + EventBridge + SM Pipeline | Alert + Task (all native Snowflake) |
| Scheduled retraining | Separate EventBridge scheduled rule + SM Pipeline | Additional Task with CRON schedule (same warehouse) |

**Result**: The entire monitoring and retraining stack -- which required 5+ AWS services -- is replaced by native Snowflake objects (Model Monitor + Alert + Tasks). Both drift-triggered and scheduled retraining coexist with no additional infrastructure.